# NeuroEvoBench Loader Example Usage

This notebook demonstrates how to use the `NeuroEvoBenchLoader` to load neuroevolution data and generate visualizations.

## Setup

First, make sure you have the required dependencies installed and the NeuroEvoBench data downloaded.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Import our custom loader
from neuroevobench_loader import NeuroEvoBenchLoader

# Set up data directory (adjust path as needed)
DATA_DIR = Path('./neuroevobench_data')  # Adjust this path

# Initialize loader
loader = NeuroEvoBenchLoader(DATA_DIR)
print("Loader initialized successfully!")

: 

## 1. List Available Datasets

The loader can scan the data directory and list all available NeuroEvoBench runs with their metadata.

In [ ]:
# List all available datasets
available = loader.list_available()
print(f"Found {len(available)} datasets")
print("\nFirst 5 datasets:")
display(available.head())

# Show unique tasks and algorithms
print(f"\nUnique tasks: {available['task'].unique()}")
print(f"Unique algorithms: {available['algorithm'].unique()}")
print(f"Available seeds: {sorted(available['seed'].unique())}")

## 2. Load a Single Run

Load data from a specific HDF5 file. The loader validates the file format and extracts metadata.

In [ ]:
# Load a specific run (adjust filename as needed)
filename = 'brax_ant_openes_seed0.h5'  # Adjust based on available files

try:
    data = loader.load_run(filename)
    print(f"Successfully loaded {filename}")
    print(f"Shape - Weights: {data['weights_3d'].shape}, Fitness: {data['fitness_2d'].shape}")
    print(f"Metadata: {data['metadata']}")
except FileNotFoundError as e:
    print(f"File not found: {e}")
    print("Available files:")
    for f in available['filename'].head():
        print(f"  {f}")

## 3. Load Multiple Seeds

For statistical analysis, you might want to load multiple seeds of the same configuration.

In [ ]:
# Load multiple seeds for the same task and algorithm
task = 'brax_ant'
algorithm = 'openes'
seeds = [0, 1, 2]  # Adjust based on available seeds

multi_data = loader.load_multiple_seeds(task, algorithm, seeds)
print(f"Loaded {len(multi_data)} runs")

# Show statistics across seeds
if multi_data:
    final_fitnesses = [np.mean(data['fitness_2d'][-1]) for data in multi_data]
    print(".3f")
    print(".3f")

## 4. Prepare Data for Visualization

Convert the loaded data to the format expected by the visualization functions. This includes computing aligned UMAP embeddings.

In [ ]:
# Prepare data for visualization
viz_data = loader.prepare_for_visualization(data)

print("Visualization data prepared:")
print(f"  Weights shape: {viz_data['weights_3d'].shape}")
print(f"  Fitness shape: {viz_data['fitness_2d'].shape}")
print(f"  Embeddings shape: {viz_data['embeddings_2d'].shape}")
print(f"  Generation IDs shape: {viz_data['generation_ids'].shape}")
print(f"  Individual IDs shape: {viz_data['individual_ids'].shape}")

## 5. Generate Aligned UMAP Visualization

Use the existing visualization functions to create publication-ready plots.

In [ ]:
# Import visualization functions
from visualizations.aligned_umap import plot

# Generate aligned UMAP plot
fig = plot(
    weights_by_gen=viz_data['weights_by_gen'],
    fitness_by_gen=viz_data['fitness_by_gen']
)

# Display the plot
plt.show()

# Save the figure (optional)
# fig.savefig('aligned_umap_example.png', dpi=300, bbox_inches='tight')

## 6. Generate Vector Field Visualization

Show the evolution dynamics with streamlines.

In [ ]:
# Import vector field visualization
from visualizations.vector_field import plot_vector_field

# Generate vector field plot
fig = plot_vector_field(
    weights_by_gen=viz_data['weights_by_gen'],
    fitness_by_gen=viz_data['fitness_by_gen'],
    generation_ids=viz_data['generation_ids'],
    individual_ids=viz_data['individual_ids']
)

# Display the plot
plt.show()

# Save the figure (optional)
# fig.savefig('vector_field_example.png', dpi=300, bbox_inches='tight')

## 7. Generate Divergence Analysis

Analyze how the population diverges over generations.

In [ ]:
# Import divergence analysis
from visualizations.divergence import compute_divergence, plot_divergence

# Compute divergence
divergence_data = compute_divergence(viz_data['embeddings_2d'])

# Generate divergence plot
fig = plot_divergence(divergence_data)

# Display the plot
plt.show()

# Save the figure (optional)
# fig.savefig('divergence_example.png', dpi=300, bbox_inches='tight')

## 8. Compute and Display Statistics

Analyze the evolution performance and embedding properties.

In [ ]:
# Compute basic statistics
fitness_over_time = np.mean(viz_data['fitness_2d'], axis=1)
embedding_spread = np.std(viz_data['embeddings_2d'], axis=(0, 1))

print("Evolution Statistics:")
print(f"  Initial fitness: {fitness_over_time[0]:.3f}")
print(f"  Final fitness: {fitness_over_time[-1]:.3f}")
print(f"  Fitness improvement: {fitness_over_time[-1] - fitness_over_time[0]:.3f}")
print(f"  Best final individual fitness: {np.max(viz_data['fitness_2d'][-1]):.3f}")

print("
Embedding Statistics:")
print(f"  Embedding spread (std): {embedding_spread}")
print(f"  Generations: {viz_data['n_generations']}")
print(f"  Population size: {viz_data['population_size']}")
print(f"  Parameters: {viz_data['n_parameters']:,}")

# Plot fitness over time
plt.figure(figsize=(10, 6))
plt.plot(fitness_over_time, 'b-', linewidth=2, alpha=0.8)
plt.fill_between(range(len(fitness_over_time)),
                 np.min(viz_data['fitness_2d'], axis=1),
                 np.max(viz_data['fitness_2d'], axis=1),
                 alpha=0.3, color='blue', label='Min-Max range')
plt.xlabel('Generation')
plt.ylabel('Average Fitness')
plt.title('Fitness Evolution Over Time')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 9. Using Embedding Caching

For large datasets, caching computed embeddings can save time on subsequent runs.

In [ ]:
# Create cache directory
cache_dir = Path('./embedding_cache')
cache_dir.mkdir(exist_ok=True)

# Prepare data with caching
viz_data_cached = loader.prepare_for_visualization(
    data,
    cache_dir=str(cache_dir),
    lambda_align=0.3,
    random_state=42
)

print("Embeddings cached for faster future loading")
print(f"Cache directory: {cache_dir}")

# List cache files
cache_files = list(cache_dir.glob("*.npy"))
print(f"Cached embedding files: {len(cache_files)}")
for f in cache_files[:3]:  # Show first 3
    print(f"  {f.name}")

## 10. Error Handling Examples

The loader includes robust error handling for common issues.

In [ ]:
# Example: Try to load non-existent file
try:
    bad_data = loader.load_run('nonexistent_file.h5')
except FileNotFoundError as e:
    print(f"Handled error: {e}")

# Example: Try to load file with wrong format
# (This would show how the loader validates file formats)

print("\nError handling examples completed.")
print("The loader provides clear error messages for:")
print("- Missing files")
print("- Corrupted HDF5 files")
print("- Invalid data shapes")
print("- Missing required datasets")

## Conclusion

This notebook demonstrated the complete workflow for using the NeuroEvoBench loader:

1. **Listing available datasets** - Scan directory and extract metadata
2. **Loading single runs** - Load and validate HDF5 files
3. **Loading multiple seeds** - For statistical analysis
4. **Preparing visualization data** - Compute aligned UMAP embeddings
5. **Generating visualizations** - Use existing plotting functions
6. **Computing statistics** - Analyze evolution performance
7. **Caching embeddings** - Speed up repeated analyses
8. **Error handling** - Robust error messages

The loader is optimized for M4 Macs with JAX Metal backend support and includes progress bars for long operations.

For production use, see `run_pipeline.py` for a command-line interface that generates all visualizations automatically.